In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# List of required descriptors
descriptor_names = [
    "SlogP_VSA2", "MolLogP", "SlogP_VSA4", "VSA_EState8", "EState_VSA1",
    "BCUT2D_LOGPLOW", "VSA_EState5", "qed", "MinEStateIndex", "TPSA",
    "SMR_VSA1", "HallKierAlpha", "NOCount", "BertzCT", "MinAbsEStateIndex",
    "NumHeteroatoms", "BCUT2D_LOGPHI", "BCUT2D_CHGHI", "VSA_EState2", "FpDensityMorgan1",
    "SMR_VSA7", "Kappa3", "SlogP_VSA5", "NumRotatableBonds", "VSA_EState4",
    "PEOE_VSA14", "SMR_VSA5", "BalabanJ", "Kappa1", "MaxPartialCharge",
    "VSA_EState3", "BCUT2D_CHGHI", "BCUT2D_MRLOW", "EState_VSA2", "MinPartialCharge",
    "BCUT2D_MWHI", "EState_VSA8", "NumHAcceptors", "PEOE_VSA7", "FpDensityMorgan3", "PEOE_VSA10"
]

def compute_descriptors(smiles):
    """Computes RDKit descriptors for a given SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Invalid SMILES

    # Compute all descriptors
    descriptor_values = {}
    for desc in descriptor_names:
        try:
            descriptor_values[desc] = rdMolDescriptors.Properties.GetProperty(desc)(mol)
        except:
            descriptor_values[desc] = None  # In case a descriptor is missing

    return descriptor_values

# Load SMILES from an Excel file
df = pd.read_excel("valid_smiles.xlsx")

# Compute descriptors for each SMILES
descriptor_data = []
for idx, row in df.iterrows():
    smiles = row["SMILES"]
    descriptors = compute_descriptors(smiles)
    if descriptors:
        descriptor_data.append({"SMILES": smiles, **descriptors})

# Convert to DataFrame and save
df_descriptors = pd.DataFrame(descriptor_data)
df_descriptors.to_excel("computed_descriptors.xlsx", index=False)

print("✅ Descriptors computed and saved in 'computed_descriptors.xlsx'")


✅ Descriptors computed and saved in 'computed_descriptors.xlsx'


In [2]:
from rdkit.Chem import Descriptors

print("Available RDKit Descriptors:", dir(Descriptors))


Available RDKit Descriptors: ['AUTOCORR2D_1', 'AUTOCORR2D_10', 'AUTOCORR2D_100', 'AUTOCORR2D_101', 'AUTOCORR2D_102', 'AUTOCORR2D_103', 'AUTOCORR2D_104', 'AUTOCORR2D_105', 'AUTOCORR2D_106', 'AUTOCORR2D_107', 'AUTOCORR2D_108', 'AUTOCORR2D_109', 'AUTOCORR2D_11', 'AUTOCORR2D_110', 'AUTOCORR2D_111', 'AUTOCORR2D_112', 'AUTOCORR2D_113', 'AUTOCORR2D_114', 'AUTOCORR2D_115', 'AUTOCORR2D_116', 'AUTOCORR2D_117', 'AUTOCORR2D_118', 'AUTOCORR2D_119', 'AUTOCORR2D_12', 'AUTOCORR2D_120', 'AUTOCORR2D_121', 'AUTOCORR2D_122', 'AUTOCORR2D_123', 'AUTOCORR2D_124', 'AUTOCORR2D_125', 'AUTOCORR2D_126', 'AUTOCORR2D_127', 'AUTOCORR2D_128', 'AUTOCORR2D_129', 'AUTOCORR2D_13', 'AUTOCORR2D_130', 'AUTOCORR2D_131', 'AUTOCORR2D_132', 'AUTOCORR2D_133', 'AUTOCORR2D_134', 'AUTOCORR2D_135', 'AUTOCORR2D_136', 'AUTOCORR2D_137', 'AUTOCORR2D_138', 'AUTOCORR2D_139', 'AUTOCORR2D_14', 'AUTOCORR2D_140', 'AUTOCORR2D_141', 'AUTOCORR2D_142', 'AUTOCORR2D_143', 'AUTOCORR2D_144', 'AUTOCORR2D_145', 'AUTOCORR2D_146', 'AUTOCORR2D_147', 'AUTO

In [2]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED, GraphDescriptors, AllChem

def compute_descriptors(mol):
    desc = {}
    
    # Compute Gasteiger charges
    try:
        AllChem.ComputeGasteigerCharges(mol)
        charges = [atom.GetDoubleProp('_GasteigerCharge') for atom in mol.GetAtoms() if atom.HasProp('_GasteigerCharge')]
        desc['MaxPartialCharge'] = max(charges) if charges else None
        desc['MinPartialCharge'] = min(charges) if charges else None
    except:
        desc['MaxPartialCharge'] = None
        desc['MinPartialCharge'] = None

    # BCUT2D descriptors
    try:
        bcut = rdMolDescriptors.BCUT2D(mol)
        desc.update({
            'BCUT2D_LOGPLOW': bcut[5],
            'BCUT2D_LOGPHI': bcut[4],
            'BCUT2D_CHGHI': bcut[2],
            'BCUT2D_MRLOW': bcut[7],
            'BCUT2D_MWHI': bcut[0]
        })
    except:
        desc.update({k: None for k in ['BCUT2D_LOGPLOW', 'BCUT2D_LOGPHI', 
                                      'BCUT2D_CHGHI', 'BCUT2D_MRLOW', 'BCUT2D_MWHI']})

    # SlogP_VSA descriptors
    try:
        slogp_vsa = rdMolDescriptors.CalcSlogPVSA(mol)
        desc.update({
            'SlogP_VSA2': slogp_vsa[1],
            'SlogP_VSA4': slogp_vsa[3],
            'SlogP_VSA5': slogp_vsa[4]
        })
    except:
        desc.update({k: None for k in ['SlogP_VSA2', 'SlogP_VSA4', 'SlogP_VSA5']})

    # EState_VSA descriptors
    try:
        estate_vsa = rdMolDescriptors.CalcEStateVSA(mol)
        desc.update({
            'VSA_EState8': estate_vsa[7],
            'EState_VSA1': estate_vsa[0],
            'VSA_EState5': estate_vsa[4],
            'VSA_EState2': estate_vsa[1],
            'VSA_EState4': estate_vsa[3],
            'VSA_EState3': estate_vsa[2],
            'EState_VSA8': estate_vsa[7],
            'EState_VSA2': estate_vsa[1]
        })
    except:
        desc.update({k: None for k in ['VSA_EState8', 'EState_VSA1', 'VSA_EState5',
                                      'VSA_EState2', 'VSA_EState4', 'VSA_EState3',
                                      'EState_VSA8', 'EState_VSA2']})

    # SMR_VSA descriptors
    try:
        smr_vsa = rdMolDescriptors.CalcSMRVSA(mol)
        desc.update({
            'SMR_VSA1': smr_vsa[0],
            'SMR_VSA7': smr_vsa[6],
            'SMR_VSA5': smr_vsa[4]
        })
    except:
        desc.update({k: None for k in ['SMR_VSA1', 'SMR_VSA7', 'SMR_VSA5']})

    # PEOE_VSA descriptors
    try:
        peoe_vsa = rdMolDescriptors.CalcPEOEVSA(mol)
        desc.update({
            'PEOE_VSA14': peoe_vsa[13],
            'PEOE_VSA7': peoe_vsa[6],
            'PEOE_VSA10': peoe_vsa[9]
        })
    except:
        desc.update({k: None for k in ['PEOE_VSA14', 'PEOE_VSA7', 'PEOE_VSA10']})

    # Morgan fingerprints
    try:
        fp1 = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 1, 2048)
        fp3 = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 3, 2048)
        desc.update({
            'FpDensityMorgan1': fp1.GetNumOnBits()/2048,
            'FpDensityMorgan3': fp3.GetNumOnBits()/2048
        })
    except:
        desc.update({k: None for k in ['FpDensityMorgan1', 'FpDensityMorgan3']})

    # Basic descriptors
    basic_desc = {
        'MolLogP': Descriptors.MolLogP,
        'qed': QED.qed,
        'MinEStateIndex': Descriptors.MinEStateIndex,
        'TPSA': Descriptors.TPSA,
        'HallKierAlpha': Descriptors.HallKierAlpha,
        'NOCount': Descriptors.NOCount,
        'BertzCT': Descriptors.BertzCT,
        'MinAbsEStateIndex': Descriptors.MinAbsEStateIndex,
        'NumHeteroatoms': Descriptors.NumHeteroatoms,
        'Kappa3': GraphDescriptors.Kappa3,
        'NumRotatableBonds': Descriptors.NumRotatableBonds,
        'BalabanJ': GraphDescriptors.BalabanJ,
        'Kappa1': GraphDescriptors.Kappa1,
        'NumHAcceptors': Descriptors.NumHAcceptors
    }

    for name, func in basic_desc.items():
        try:
            desc[name] = func(mol)
        except:
            desc[name] = None

    return desc

# Read data and process
df = pd.read_excel('valid_smiles.xlsx', sheet_name='Sheet1')

# Get descriptor keys from a dummy molecule
dummy_mol = Chem.MolFromSmiles('C')
descriptor_keys = list(compute_descriptors(dummy_mol).keys())

results = []
for smi in df['SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        results.append(compute_descriptors(mol))
    else:
        results.append({k: None for k in descriptor_keys})

df_result = pd.DataFrame(results)
df_final = pd.concat([df, df_result], axis=1)
df_final.to_excel('descriptors_output.xlsx', index=False)